# Notebook 1 — One-Turn LLM Patterns

In this notebook we'll build the foundations of LLM-powered features for our nanny agency:

1. **Email generation** with prompts — zero-shot → role-conditioned → few-shot → structured output
2. **Personalized birthday wishes** — system/user split, personalization variables
3. **PDF extraction** — turn unstructured resumes/intakes into validated Pydantic records
4. **Embeddings + matching** — vector search in ChromaDB with a 2D UMAP visualization

Total time: ~1h 15m. Each section ends with a **Try it** cell — feel free to tweak prompts, models, and parameters. The on-disk cache (`CachedOpenAI`) means re-runs cost nothing.

**Outputs of this notebook:** the `nanny_db/` Chroma collection, used by Notebook 2.

## 0. Setup & env check

Loads the OpenAI key, sets the repo root so imports work whether you launched Jupyter from the repo root or from `notebooks/`, and runs a one-call smoke test.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# When jupyter is launched from the repo root, the cwd here is notebooks/.
# Add the repo root so we can import the nanny_workshop package and baml_client.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")
key = os.getenv("OPENAI_API_KEY")
assert key and key.startswith("sk-"), "Set OPENAI_API_KEY in .env (see README step 3)."

from nanny_workshop.openai_client import CachedOpenAI

# Disk cache lets you re-run this notebook for free.
client = CachedOpenAI(cache_dir=ROOT / ".cache" / "n1")

# Smoke test: one round trip to OpenAI.
reply = client.complete(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Reply with one word: ready"}],
)
print(f"Setup OK — model reply: {reply!r}")

## 1. Email generation with prompts

The same task — drafting a booking confirmation email — produces wildly different output depending on how you prompt for it. We'll walk through four progressively-constrained prompt styles and see how output quality and shape change.

The scenario: the agency needs to confirm a booking with a parent. Let's see what the LLM does with increasing amounts of guidance.

In [ ]:
# 1a. ZERO-SHOT: give the model the bare task and see what it does.
# Notice: no role, no tone guidance, no format. We're trusting the model's defaults.

zero_shot_prompt = "Write an email confirming a nanny booking for next Thursday."

zero_shot = client.complete(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": zero_shot_prompt}],
)
print(zero_shot)

In [ ]:
# 1b. ROLE + STYLE: tell the model who it is and how to sound.
# Setting persona ("warm, professional") makes outputs much more consistent across runs.

role_styled = client.complete(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You are the Nanny Agency's customer-care assistant. "
                "Tone: warm, professional, reassuring. "
                "Keep emails to 4-6 sentences. Sign as 'The Nanny Agency Team'."
            ),
        },
        {
            "role": "user",
            "content": "Write a booking confirmation email for the Johnson family with nanny Maria on Thursday for 6 hours.",
        },
    ],
)
print(role_styled)

In [ ]:
# 1c. FEW-SHOT: show the model 2 example emails first.
# This teaches output STRUCTURE by demonstration, which is often clearer than instructions.

few_shot_examples = [
    {
        "role": "user",
        "content": "Confirm booking — family: Chen, nanny: Aisha, day: Tuesday, hours: 4.",
    },
    {
        "role": "assistant",
        "content": (
            "Subject: Your Tuesday booking is confirmed\n\n"
            "Hi Chen family,\n\n"
            "We're delighted to confirm Aisha will be with you this Tuesday for 4 hours. "
            "She'll arrive 10 minutes before the start time. Please reply if anything changes.\n\n"
            "Warmly,\nThe Nanny Agency Team"
        ),
    },
    {
        "role": "user",
        "content": "Confirm booking — family: Patel, nanny: Sam, day: Saturday, hours: 5.",
    },
    {
        "role": "assistant",
        "content": (
            "Subject: Saturday with Sam — confirmed\n\n"
            "Hi Patel family,\n\n"
            "All set: Sam will be with you Saturday for 5 hours. "
            "Looking forward to a great session — reach out if anything comes up.\n\n"
            "Warmly,\nThe Nanny Agency Team"
        ),
    },
]

few_shot = client.complete(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You write booking confirmation emails in the format shown."},
        *few_shot_examples,
        {"role": "user", "content": "Confirm booking — family: Johnson, nanny: Maria, day: Thursday, hours: 6."},
    ],
)
print(few_shot)

In [ ]:
# 1d. STRUCTURED OUTPUT: ask for JSON so a downstream system can parse it.
# Use response_format={"type": "json_object"} so the model is constrained to valid JSON.

import json

structured = client.complete(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You return ONLY a JSON object with fields: subject (string), greeting (string), "
                "body (string, 2-4 sentences), signature (string). No prose outside JSON."
            ),
        },
        {
            "role": "user",
            "content": "Confirm Johnson family booking with Maria, Thursday, 6 hours.",
        },
    ],
    response_format={"type": "json_object"},
)

parsed = json.loads(structured)
print(json.dumps(parsed, indent=2))
print()
print(f"Type: {type(parsed).__name__}, keys: {list(parsed.keys())}")

In [ ]:
# 🎯 TRY IT: change ONE thing and observe.
#   - Change the tone in the system prompt to "concise and matter-of-fact"
#   - Or swap gpt-4o-mini for gpt-4o
#   - Or set temperature to 1.5 (vs default 0.0) — note temperature is a CachedOpenAI arg
#
# Each different (model, messages, temperature) combo creates a new cache entry,
# so you can compare without re-paying for earlier runs.

your_system = "You are the Nanny Agency's customer-care assistant. Tone: warm, professional, reassuring."
your_user = "Write a booking confirmation email for the Johnson family with nanny Maria on Thursday for 6 hours."

experiment = client.complete(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": your_system},
        {"role": "user", "content": your_user},
    ],
    temperature=0.0,
)
print(experiment)